# Module 9: Training Sets

**Snowpark ML Fundamentals - Week 2 | Lunch & Learn**

---

## Learning Objectives
- Create a spine DataFrame for training set generation
- Generate point-in-time correct training sets
- Retrieve feature values for inference
- Understand why point-in-time joins matter

## Key Concept
> **Point-in-time correct** training sets prevent data leakage by only joining
> features that were available at the time of the training example.
> This is the core value proposition of a Feature Store.

---

In [ ]:
%load_ext autoreload
%autoreload 2

## 9.1 Setup

This notebook assumes you've run notebook 08 (Feature Views) to register
the entities and feature views.

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.feature_store.entities import setup_feature_store

session = create_session(env_path='../.env')
fs = setup_feature_store(session)
print("Feature Store initialized.")

## 9.2 The Spine DataFrame

A **spine** defines which entity keys (and optionally timestamps) to
look up features for:

```
CUSTOMER_ID  |  EVENT_TIMESTAMP
-------------|------------------
1            |  2024-01-15
2            |  2024-02-01
3            |  2024-01-20
```

The Feature Store joins features available **at or before** each timestamp.

In [ ]:
from snowpark_fundamentals.feature_store.training_sets import create_spine_dataframe

# Simple spine: all unique customer IDs (no timestamp)
spine_df = create_spine_dataframe(
    session,
    entity_table="CUSTOMER_RFM_FEATURES",
    entity_key="CUSTOMER_ID",
)

print(f"Spine rows: {spine_df.count()}")
spine_df.show(5)

## 9.3 Generate a Training Set

The training set joins the spine with registered FeatureViews.
This is the recommended way to build training data — it ensures
every model uses consistent, governed features.

In [ ]:
from snowpark_fundamentals.feature_store.training_sets import generate_training_set
from snowpark_fundamentals.feature_store.feature_views import get_feature_view

# Get registered feature views
rfm_fv = get_feature_view(fs, "CUSTOMER_RFM_FV", "V1")
behavioral_fv = get_feature_view(fs, "CUSTOMER_BEHAVIORAL_FV", "V1")

# Generate training set with both feature views
training_set = generate_training_set(
    fs=fs,
    spine_df=spine_df,
    features=[rfm_fv, behavioral_fv],
    name="CHURN_TRAINING_SET",
)

# Convert to DataFrame for inspection
training_df = training_set.read.to_snowpark_dataframe()
print(f"Training set: {training_df.count()} rows, {len(training_df.columns)} columns")
print(f"Columns: {training_df.columns}")

In [ ]:
training_df.show(5)

## 9.4 Retrieve Feature Values for Inference

At inference time, use `retrieve_feature_values()` to look up the
latest features for a set of entity keys.

In [ ]:
from snowpark_fundamentals.feature_store.training_sets import retrieve_feature_values

# Simulate inference: look up features for a few customers
inference_spine = spine_df.limit(10)

feature_values = retrieve_feature_values(
    fs=fs,
    spine_df=inference_spine,
    features=[rfm_fv, behavioral_fv],
)

print(f"Feature values for inference: {feature_values.count()} rows")
feature_values.show(5)

## 9.5 Why Point-in-Time Matters

Without point-in-time joins, your training set can contain **future information**:

```
BAD:  Training on Jan 15 data with features computed from Feb data
      -> Model learns patterns it can't see at inference time
      -> Artificially inflated metrics, poor production performance

GOOD: Training on Jan 15 data with features computed BEFORE Jan 15
      -> Model learns from the same signals available at inference
      -> Honest metrics, consistent production performance
```

The Feature Store handles this automatically when your spine includes
a timestamp column.

## Key Takeaways

1. A **spine DataFrame** defines which entity keys to look up
2. `generate_training_set()` joins features from registered FeatureViews
3. `retrieve_feature_values()` fetches latest features for inference
4. **Point-in-time correctness** prevents data leakage in training
5. The Feature Store ensures **training-serving consistency**

---

**Next: [dbt Feature Store Guide](../dbt_feature_store/README.md)** — Complete dbt command reference, model walkthrough, and how to build your own features.

In [ ]:
session.close()